# பாடம் 01 - AI முகவர்களுக்கு அறிமுகம்

**புதியவர்களுக்கு AI முகவர்கள்** பாட தொகுப்பின் முதல் பாடத்திற்கு வரவேற்கிறோம்!

**AI முகவர்** என்பது ஒரு பெரிய மொழி மாதிரியை (LLM) அதன் காரணமளிக்கும் இயந்திரமாக பயன்படுத்தும் ஒரு நிரல் ஆகும், மேலும் அது *நிகழ்வுகள்* ஐ நடமಾಡவல்லது — APIs ஐ அழைப்பது, தரவுத்தளங்களை வினவுவது, அல்லது குறியீட்டை இயக்குவது — பயனருக்காக ஒரு இலக்கை அடைய.

இந்த நோட்புக் இல் நீங்கள் உங்கள் முதல் முகவரான ஒரு **பயண முகவர்** ஐ உருவாக்கப் போகிறீர்கள், இது விடுமுறை தலங்களை பரிந்துரைக்கும். இதைவிட கீழ்காணும் விஷயங்களை நீங்கள் கற்றுக்கொள்வீர்கள்:

1. **Microsoft Agent Framework** பயன்படுத்தி Azure AI Foundry Agent சேவையை இணைப்பது.
2. முகவருக்கு ஒரு **கருவி** கொடுத்தல் — அது அழைக்கக்கூடிய சாதாரண Python செயல்பாடு.
3. முகவரை இயக்கி அதன் பதிலை ஆய்வு செய்வது.
4. முகவரின் பதிலை டோக்கன்-வழியாக ஸ்ட்ரீம் செய்வது.


## அமைத்தல்

இந்த நோட்புக் ஓடுவதற்கு முன்னர், நீங்கள் எனக்கு உறுதிப்படுத்திக் கொள்ள வேண்டும்:

1. **ஒரு Azure AI Foundry திட்டம்** மற்றும் அதில் பயன்பாட்டுப் படுத்தப்பட்ட உரையாடல் மாடல் (உதா. `gpt-4o-mini`).
2. **Azure CLI-யில் உள்நுழைந்திருத்தல்** — உங்கள் டெர்மினலில் `az login` என இயக்கவும்.
3. **தேவையான சூழல் மாறிகள் அமைத்தல்:**
   - `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Azure AI Foundry திட்டத்தின் இறுதி புள்ளி.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — நீங்கள் தயாரித்த மாடலின் பெயர்.

கீழே உள்ள செலல் நீங்கள் தேவையான Python தொகுதிகள் (package) நிறுவும்.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## உங்கள் முதல் முகவரியை உருவாக்கல்

ஒரு முகவரிக்கு இரண்டு விஷயங்கள் தேவை:

- அது *யார்* மற்றும் *எப்படி நடக்க வேண்டும்* என்பதைக் கூறும் **வழிமுறைகள்** (ஒரு கணினி உத்வேகம்).
- முகவரி தகவலைப் பெற அல்லது நடவடிக்கைகள் செய்ய அழைக்கக்கூடிய `@tool` கொண்டு அலங்கரிக்கப்பட்ட பைதான் செயல்பாடுகள் என்ற **கருவிகள்**.

கீழே, பிரபலமான விடுமுறை இடங்களின் பட்டியலை வழங்கும் ஒரு எளிய கருவி வரையறுக்கப்பட்டுள்ளது. பயனாளர் பயண பரிந்துரைகளை கேட்கும் போது முகவரி இந்த கருவியை பயன்படுத்தும்.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ஸ்ட்ரீமிங் பதில்கள்

ஒரு மேலதிக இடையூறற்ற அனுபவத்திற்கு நீங்கள் முகவரியின் பதிலை **ஸ்ட்ரீம்** செய்யலாம். முழு பதிலை காத்திருக்காமல், முகவர் உருவாக்கும் போதே உரை துண்டுகளை வழங்குகிறான். இந்த வழி, ஆவணங்களை நேரடி நேரத்தில் காட்ட விரும்பும் உரையாடல் இடைமுகங்களில் மிகவும் பயனுள்ளதாக இருக்கும்.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் கற்றுக்கொண்டது:

- **AzureAIProjectAgentProvider** மூலம் Azure AI Foundry Agent சேவைக்கு இணைவதற்கான மற்றொரு வழங்குபவரை உருவாக்குவது.
- முகவர் உங்கள் Python செயல்பாடுகளை அழைக்க `@tool` அலங்காரியைப் பயன்படுத்தி கருவியை வரையறுக்கவும்.
- பயனர் செய்தியுடன் முகவரை இயக்கி அதன் பதிலை அச்சிடவும்.
- நேரடி வெளியீட்டுக்கு பதில்களை ஸ்ட்ரீம் செய்யவும்.

அடுத்த பாடத்தில் நாங்கள் முகவர் அமைப்புகளை விரிவாக ஆராய்ந்து முகவர்களுக்கு அதிக சக்திவாய்ந்த கருவிகள் மற்றும் பல படி காரணிக்கும் திறன்களை வழங்க கற்றுக்கொள்ளப்போகிறோம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**அறிவிப்பு**:  
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையான [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்கிறோம் என்றாலும், தானாக செய்யப்பட்ட மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருப்பதற்கான வாய்ப்பு இருக்கிறது. அடிப்படையான ஆவணம் அதன் தாய்மை மொழியில் ஜீரணமாகவும் அதிகாரபூர்வமாகவும் கருதப்பட வேண்டும். மிக முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படக்கூடிய எந்த தவறான புரிதலும் அல்லது பொருள்படுத்தல்களுக்கும் நாங்கள் பொறுப்பல்ல.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
